# Tune the argument NULL-bias (recall vs precision)

Sweeps the args model's **NULL-bias** to pick a recall-friendlier default, and shows its effect on real sentences.

**What NULL-bias does:** for each span the detection head proposes, the role head decides a role or `NULL` (reject). Higher bias → reject more (higher precision, fewer args); lower/negative → keep more (higher recall, more peripheral roles like `Place`/`Time`).

**Important limitation:** NULL-bias only re-scores spans the **detection head already proposed**. A role that was never *detected* (e.g. a span the detector missed entirely) cannot be recovered by any NULL-bias. So this lever recovers *rejected* args, not *undetected* ones.

Same prerequisites as `parse_de.ipynb` (re-made zip incl. `salsa_pipeline.py` + corpus; trained models on Drive).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
ZIP = "/content/drive/MyDrive/texture_frames_colab.zip"
assert os.path.exists(ZIP), f"Upload the project zip to {ZIP}."
!rm -rf /content/Texture_Frames && mkdir -p /content/Texture_Frames
!unzip -q "$ZIP" -d /content/Texture_Frames
FRAME_DIR = "/content/drive/MyDrive/Texture_Frames/models/frame2_de"
ARGS_DIR  = "/content/drive/MyDrive/Texture_Frames/models/args2_de"
assert os.path.exists(os.path.join(ARGS_DIR, "args2_model.pt")), "args checkpoint missing on Drive"

In [ ]:
!pip install -q "transformers==4.57.6" "huggingface_hub<1.0" sentencepiece simplemma "numpy>=2"

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")
sys.path.insert(0, "/content/Texture_Frames/German_parser")
os.chdir("/content/Texture_Frames/German_parser")
import torch, transformers; print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
from huggingface_hub import __version__ as _hv
assert int(_hv.split(".")[0]) < 1, (
    f"huggingface_hub {_hv} is loaded, but transformers 4.57.6 needs <1.0 \u2014 "
    "do Runtime \u2192 Restart session, then Run all from the top.")

## Load the parser

In [ ]:
from salsa_pipeline import GermanFrameParser
parser = GermanFrameParser(frame_dir=FRAME_DIR, args_dir=ARGS_DIR)
print("ready — device:", parser.device)

## 1. Quantitative sweep on DEV (extended, recall-leaning grid)

Picks the NULL-bias that **maximises recall while F1 stays within 0.5 pt of the best** — a principled recall-friendly default.

In [ ]:
from train_args2_de import evaluate_args2_de, print_report

GRID = [-6.0, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 4.0]
res = evaluate_args2_de(parser.args_model, parser.args_tok, parser.lexicon,
                        parser.role2id, parser.id2role, split="dev", null_biases=GRID)
print_report(res)

bb = res["by_bias"]
best_f1 = max(v["f1"] for v in bb.values())
eligible = {b: v for b, v in bb.items() if v["f1"] >= best_f1 - 0.005}
rec = max(eligible, key=lambda b: eligible[b]["recall"])
print(f"\nBest F1 = {best_f1:.3f}.  Recall-leaning pick (F1 within 0.5pt, max recall): "
      f"null_bias = {rec:+.1f}  (P={bb[rec]['precision']:.3f} R={bb[rec]['recall']:.3f} F1={bb[rec]['f1']:.3f})")

## 2. Confirm on TEST at the current (+2.0) vs the recall-leaning pick

In [ ]:
for b in sorted({2.0, rec}):
    r = evaluate_args2_de(parser.args_model, parser.args_tok, parser.lexicon,
                          parser.role2id, parser.id2role, split="test", null_biases=[b])
    m = r["by_bias"][b]
    print(f"null_bias {b:+.1f}:  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}")

## 3. Qualitative — does a lower bias recover peripheral roles?

Watch the `Place`/`Time` roles (e.g. *am Bahnhof*) appear as the bias drops. If a role never appears at *any* bias, the detector didn't propose it (beyond NULL-bias's reach).

In [ ]:
sents = [
    "Die Polizei verhaftete den Verdächtigen am Bahnhof .",
    "Die Regierung kündigte an , die Steuern zu erhöhen .",
]
for b in (2.0, 0.0, -2.0, -4.0):
    parser.null_bias = b
    print(f"\n===== null_bias = {b:+.1f} =====")
    for s in sents:
        print("#", s)
        for ann in parser.parse(s):
            roles = [(a.role, a.text) for a in ann.arguments]
            print(f"   [{ann.frame}] {ann.trigger!r}: {roles}")
parser.null_bias = 2.0   # restore

## Set your default

Once you've picked a value, construct the parser with it:
```python
parser = GermanFrameParser(frame_dir=..., args_dir=..., null_bias=0.0)   # recall-leaning
```
Or set it live: `parser.null_bias = 0.0`. Reminder: this recovers rejected spans, not undetected ones — if *am Bahnhof* never appears even at −4, the detection head missed it and NULL-bias can't help.